# **WEEK 1**
Testing multiple embedding models and multiple open-source LLMs.

# **Load Document**

In [ ]:
from google.colab import files

uploaded = files.upload()

path = list(uploaded.keys())[0]

print(path)

Saving rag_knowledge_base.txt.txt to rag_knowledge_base.txt (1).txt
rag_knowledge_base.txt (1).txt


# **EXTRACT TEXT**

In [ ]:
import re

def load_text(path):
  with open(path, 'r', encoding='utf-8') as f:
    raw_text = f.read()
  return raw_text

text = load_text(path)
print(text[:1000])

RAG Knowledge Base

1. Retrieval-Augmented Generation

Retrieval-Augmented Generation, commonly known as RAG, is a technique that combines information retrieval with large language model generation. Instead of relying only on information stored inside the parameters of a language model, a RAG system retrieves relevant information from an external knowledge base and provides that information as context to the language model.

A typical RAG pipeline contains several stages. First, documents are collected and divided into smaller chunks. Each chunk is converted into a numerical vector representation called an embedding. These embeddings are stored in a vector database. When a user submits a question, the question is also converted into an embedding. The system compares the query embedding with document embeddings and retrieves the most relevant chunks. The retrieved chunks are then provided to a language model, which generates the final answer.

RAG can reduce hallucinations because the l

In [ ]:
print('No of characters:', len(text))
print('No of words', len(text.split()))
print('No of lines', len(text.splitlines()))

No of characters: 11769
No of words 1666
No of lines 274


# **Data Cleaning**

In [ ]:
# !pip install unstructured

In [ ]:
import re

def clean_text(text):

    # Normalize Windows/Linux line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove tabs
    text = text.replace("\t", " ")

    # Remove trailing/leading spaces from each line
    lines = [line.strip() for line in text.split("\n")]

    # Remove completely empty lines at the beginning/end
    while lines and not lines[0]:
        lines.pop(0)

    while lines and not lines[-1]:
        lines.pop()

    # Reduce multiple spaces inside a line
    lines = [
        re.sub(r"[ ]{2,}", " ", line)
        for line in lines
    ]

    # Preserve paragraph/heading boundaries
    cleaned = "\n".join(lines)

    # Reduce 3+ consecutive blank lines to 2
    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned)

    return cleaned.strip()

In [ ]:
cleaned_document = clean_text(text)

print("----- CLEANED -----")
print(cleaned_document[:2000])

----- CLEANED -----
RAG Knowledge Base

1. Retrieval-Augmented Generation

Retrieval-Augmented Generation, commonly known as RAG, is a technique that combines information retrieval with large language model generation. Instead of relying only on information stored inside the parameters of a language model, a RAG system retrieves relevant information from an external knowledge base and provides that information as context to the language model.

A typical RAG pipeline contains several stages. First, documents are collected and divided into smaller chunks. Each chunk is converted into a numerical vector representation called an embedding. These embeddings are stored in a vector database. When a user submits a question, the question is also converted into an embedding. The system compares the query embedding with document embeddings and retrieves the most relevant chunks. The retrieved chunks are then provided to a language model, which generates the final answer.

RAG can reduce hallucin

In [ ]:
print("Original characters:", len(text))
print("Cleaned characters:", len(cleaned_document))

print("Original words:", len(text.split()))
print("Cleaned words:", len(cleaned_document.split()))

print("Original paragraphs:", len(re.split(r"\n\s*\n", text.strip())))
print("Cleaned paragraphs:", len(re.split(r"\n\s*\n", cleaned_document.strip())))

Original characters: 11769
Cleaned characters: 11754
Original words: 1666
Cleaned words: 1666
Original paragraphs: 105
Cleaned paragraphs: 105


In [ ]:
with open('cleaned_rag_document.txt', 'w', encoding = 'utf-8') as f:
  f.write(cleaned_document)

print("Cleaned Doc saved.....")

Cleaned Doc saved.....


# **Creating Chunks**

In [ ]:
def create_chunks(text, chunk_size = 500, overlap = 100):
  start = 0
  chunks = []

  while start < len(text):
    end = start + chunk_size
    chunks.append(text[start:end])

    start += chunk_size - overlap

  return chunks

In [ ]:
chunks = create_chunks(cleaned_document)

print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk)

Number of chunks: 30

--- Chunk 1 ---
RAG Knowledge Base

1. Retrieval-Augmented Generation

Retrieval-Augmented Generation, commonly known as RAG, is a technique that combines information retrieval with large language model generation. Instead of relying only on information stored inside the parameters of a language model, a RAG system retrieves relevant information from an external knowledge base and provides that information as context to the language model.

A typical RAG pipeline contains several stages. First, documents are co

--- Chunk 2 ---
text to the language model.

A typical RAG pipeline contains several stages. First, documents are collected and divided into smaller chunks. Each chunk is converted into a numerical vector representation called an embedding. These embeddings are stored in a vector database. When a user submits a question, the question is also converted into an embedding. The system compares the query embedding with document embeddings and retrieves the most

# **Embedding**

In [ ]:
from sentence_transformers import SentenceTransformer

model_LM = SentenceTransformer('all-MiniLM-L6-v2')
model_MP = SentenceTransformer('all-mpnet-base-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Helper Functions for Embedding and FAISS Indexing

To improve code modularity and reusability, I'll define functions for generating embeddings from text chunks and for creating a FAISS index from these embeddings.

In [ ]:
import numpy as np
import faiss

def get_embeddings(model, chunks):
    """
    Generates embeddings for a list of text chunks using a given SentenceTransformer model.

    Args:
        model: A loaded SentenceTransformer model.
        chunks (list): A list of text strings (chunks).

    Returns:
        np.ndarray: A NumPy array containing the embeddings.
    """
    print(f"Generating embeddings using {model._get_name()}...")
    embeddings = model.encode(chunks, show_progress_bar=True, convert_to_numpy=True)
    print(f"Embeddings shape: {embeddings.shape}")
    return embeddings

# **FAISS Embeddings**

In [ ]:
def create_faiss_index(embeddings):
    """
    Creates and populates a FAISS IndexFlatL2 index with the given embeddings.

    Args:
        embeddings (np.ndarray): A NumPy array of embeddings.

    Returns:
        faiss.IndexFlatL2: A FAISS index object.
    """
    print(f"Creating FAISS index with {embeddings.shape[0]} vectors of dimension {embeddings.shape[1]}...")
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings.astype("float32"))
    print("FAISS index created successfully.")
    return index

In [ ]:
encoded_text_MiniLM = get_embeddings(model_LM, chunks)
encoded_text_MP = get_embeddings(model_MP, chunks)

Generating embeddings using SentenceTransformer...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape: (30, 384)
Generating embeddings using SentenceTransformer...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape: (30, 768)


In [ ]:
print(f"Type of encoded_text_MiniLM: {type(encoded_text_MiniLM)}")
print(f"Type of encoded_text_MP: {type(encoded_text_MP)}")

Type of encoded_text_MiniLM: <class 'numpy.ndarray'>
Type of encoded_text_MP: <class 'numpy.ndarray'>


In [ ]:
print(f'Shape of MiniLM: {encoded_text_MiniLM.shape}')
print(f'Shape of MP: {encoded_text_MP.shape}')

Shape of MiniLM: (30, 384)
Shape of MP: (30, 768)


# **FAISS Index for MiniLM**

In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 69.2 MB/s eta 0:00:00


In [ ]:
index_LM = create_faiss_index(encoded_text_MiniLM)

print("No of vectors:", index_LM.ntotal)

Creating FAISS index with 30 vectors of dimension 384...
FAISS index created successfully.
No of vectors: 30


In [ ]:
k = 3

distance, indices = retrieve(index_LM, quest_embedding_LM, k)

print(distance)
print(indices)

[0.8161324 0.8347411 0.904704 ]
[0 3 2]


In [ ]:
for i, rank_idx in enumerate(indices[0]):
  print(f'Rank {i + 1}:')
  print(chunks[rank_idx])

Rank 1:
RAG Knowledge Base

1. Retrieval-Augmented Generation

Retrieval-Augmented Generation, commonly known as RAG, is a technique that combines information retrieval with large language model generation. Instead of relying only on information stored inside the parameters of a language model, a RAG system retrieves relevant information from an external knowledge base and provides that information as context to the language model.

A typical RAG pipeline contains several stages. First, documents are co
Rank 2:
ncorrect answer based on the retrieved information.

The quality of a RAG system depends on several factors, including document preprocessing, chunk size, chunk overlap, embedding quality, retrieval strategy, number of retrieved documents, context construction, and language model capability.

2. Document Chunking

Document chunking is the process of dividing a large document into smaller sections that can be processed by an embedding model and retrieved independently.

A documen

# **FAISS Index for MPNet**

In [ ]:
index_MP = create_faiss_index(encoded_text_MP)

print("No of vectors:", index_MP.ntotal)

Creating FAISS index with 30 vectors of dimension 768...
FAISS index created successfully.
No of vectors: 30


In [ ]:
distance, indices = retrieve(index_MP, quest_embedding_MP, k)

print(distance)
print(indices)

[0.87721646 0.9886872  1.1587169 ]
[ 0  3 11]


In [ ]:
for i, rank_idx in enumerate(indices[0]):
  print(f'Rank {i + 1}:')
  print(chunks[rank_idx])

Rank 1:
RAG Knowledge Base

1. Retrieval-Augmented Generation

Retrieval-Augmented Generation, commonly known as RAG, is a technique that combines information retrieval with large language model generation. Instead of relying only on information stored inside the parameters of a language model, a RAG system retrieves relevant information from an external knowledge base and provides that information as context to the language model.

A typical RAG pipeline contains several stages. First, documents are co
Rank 2:
ncorrect answer based on the retrieved information.

The quality of a RAG system depends on several factors, including document preprocessing, chunk size, chunk overlap, embedding quality, retrieval strategy, number of retrieved documents, context construction, and language model capability.

2. Document Chunking

Document chunking is the process of dividing a large document into smaller sections that can be processed by an embedding model and retrieved independently.

A documen

# **Testing on Questions**

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving phase 1 questions.docx to phase 1 questions.docx


In [ ]:
!pip install -q python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 19.3 MB/s eta 0:00:00


In [ ]:
from docx import Document

file_name = list(uploaded.keys())[0]

doc = Document(file_name)

questions = []

table = doc.tables[0]

for row in table.rows[1:]:
    cells = row.cells

    questions.append({
        "id": cells[0].text.strip(),
        "question": cells[1].text.strip(),
        "expected_section": cells[2].text.strip()
    })

print("Number of questions:", len(questions))

for q in questions[:5]:
    print(q)

Number of questions: 30
{'id': 'Q01', 'question': 'What is Retrieval-Augmented Generation?', 'expected_section': 'RAG'}
{'id': 'Q02', 'question': 'How does RAG use an external knowledge base?', 'expected_section': 'RAG'}
{'id': 'Q03', 'question': 'What are the main stages of a typical RAG pipeline?', 'expected_section': 'RAG'}
{'id': 'Q04', 'question': 'Why can RAG reduce hallucinations?', 'expected_section': 'RAG'}
{'id': 'Q05', 'question': 'Does RAG guarantee that generated answers are correct?', 'expected_section': 'RAG'}


In [ ]:
section_names = [
    "Retrieval-Augmented Generation",
    "Document Chunking",
    "Text Embeddings",
    "Vector Similarity",
    "Corrective Retrieval-Augmented Generation",
    "Self-RAG",
    "GraphRAG",
    "ReAct",
    "Large Language Models",
    "Hallucination in Large Language Models",
    "Evidence and Hallucination Detection",
    "Natural Language Inference",
    "Evidence Graph",
    "Trust Score",
    "Limitations of Existing RAG Approaches",
    "Proposed Research Direction"
]

In [ ]:
section_map = {
    "RAG": "Retrieval-Augmented Generation",
    "Document Chunking": "Document Chunking",
    "Text Embeddings": "Text Embeddings",
    "Vector Similarity": "Vector Similarity",
    "CRAG": "Corrective Retrieval-Augmented Generation",
    "Self-RAG": "Self-RAG",
    "GraphRAG": "GraphRAG",
    "ReAct": "ReAct",
    "Hallucination": "Hallucination in Large Language Models",
    "NLI": "Natural Language Inference",
    "Evidence Graph": "Evidence Graph",
    "Trust Score": "Trust Score"
}

In [ ]:
query_texts = [q["question"] for q in questions]

query_embeddings_LM = model_LM.encode(
    query_texts,
    convert_to_numpy=True
)

query_embeddings_MP = model_MP.encode(
    query_texts,
    convert_to_numpy=True
)

print(query_embeddings_LM.shape)
print(query_embeddings_MP.shape)

(30, 384)
(30, 768)


In [ ]:
faiss.IndexFlatL2()

<faiss.swigfaiss.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x793cefbae570> >

In [ ]:
import faiss

index_LM = create_faiss_index(encoded_text_MiniLM)
index_MP = create_faiss_index(encoded_text_MP)

print("MiniLM vectors:", index_LM.ntotal)
print("MPNet vectors:", index_MP.ntotal)

Creating FAISS index with 30 vectors of dimension 384...
FAISS index created successfully.
Creating FAISS index with 30 vectors of dimension 768...
FAISS index created successfully.
MiniLM vectors: 30
MPNet vectors: 30


In [ ]:
def retrieve(index, query_embedding, k):
    query_embedding = query_embedding.reshape(1, -1).astype("float32")

    distances, indices = index.search(query_embedding, k)

    return distances[0], indices[0]

In [ ]:
def evaluate_question(
    question,
    expected_section,
    index,
    query_embedding,
    k=5
):

    distances, indices = retrieve(
        index,
        query_embedding,
        k
    )

    expected_section = section_map.get(
        expected_section,
        expected_section
    )

    hits = []

    for idx in indices:
        chunk_text = chunks[idx]

        found = expected_section.lower() in chunk_text.lower()

        hits.append(found)

    return {
        "retrieved_indices": indices.tolist(),
        "distances": distances.tolist(),
        "hit@1": hits[0],
        "hit@3": any(hits[:3]),
        "hit@5": any(hits[:5])
    }

# **Evaluate MiniLM**

In [ ]:
results_minilm = []

for i, q in enumerate(questions):

    result = evaluate_question(
        q["question"],
        q["expected_section"],
        index_LM,
        query_embeddings_LM[i],
        k=5
    )

    results_minilm.append({
        "id": q["id"],
        "question": q["question"],
        "expected_section": q["expected_section"],
        **result
    })

In [ ]:
total = len(results_minilm)

hit1 = sum(r["hit@1"] for r in results_minilm)
hit3 = sum(r["hit@3"] for r in results_minilm)
hit5 = sum(r["hit@5"] for r in results_minilm)

print("MiniLM Retrieval Evaluation")
print("----------------------------")

print("Total Questions:", total)

print("Hit@1:", hit1 / total)
print("Hit@3:", hit3 / total)
print("Hit@5:", hit5 / total)

MiniLM Retrieval Evaluation
----------------------------
Total Questions: 30
Hit@1: 0.43333333333333335
Hit@3: 0.6333333333333333
Hit@5: 0.7


In [ ]:
print("Hit@1:", round(hit1 / total * 100, 2), "%")
print("Hit@3:", round(hit3 / total * 100, 2), "%")
print("Hit@5:", round(hit5 / total * 100, 2), "%")

Hit@1: 43.33 %
Hit@3: 63.33 %
Hit@5: 70.0 %


In [ ]:
for r in results_minilm[:5]:

    print("\n" + "=" * 60)

    print("ID:", r["id"])
    print("Question:", r["question"])
    print("Expected:", r["expected_section"])

    print("Retrieved:", r["retrieved_indices"])

    print("Distances:", r["distances"])

    print("Hit@1:", r["hit@1"])
    print("Hit@3:", r["hit@3"])
    print("Hit@5:", r["hit@5"])


ID: Q01
Question: What is Retrieval-Augmented Generation?
Expected: RAG
Retrieved: [0, 12, 11, 29, 13]
Distances: [0.8344131708145142, 0.9144470691680908, 0.9328814148902893, 0.9853886961936951, 1.0342328548431396]
Hit@1: True
Hit@3: True
Hit@5: True

ID: Q02
Question: How does RAG use an external knowledge base?
Expected: RAG
Retrieved: [0, 2, 26, 28, 3]
Distances: [0.5612514615058899, 0.7818684577941895, 0.7957755327224731, 0.8689990639686584, 0.9147319793701172]
Hit@1: True
Hit@3: True
Hit@5: True

ID: Q03
Question: What are the main stages of a typical RAG pipeline?
Expected: RAG
Retrieved: [1, 3, 11, 0, 2]
Distances: [0.9756208062171936, 1.0740584135055542, 1.1959331035614014, 1.2502195835113525, 1.2541382312774658]
Hit@1: False
Hit@3: True
Hit@5: True

ID: Q04
Question: Why can RAG reduce hallucinations?
Expected: RAG
Retrieved: [2, 19, 20, 3, 0]
Distances: [0.8922187089920044, 0.9960722327232361, 1.3356208801269531, 1.3443355560302734, 1.4207518100738525]
Hit@1: False
Hit@3: Fa

# **Evaluate MPNet**

In [ ]:
results_mp = []

for i, q in enumerate(questions):

    result = evaluate_question(
        q["question"],
        q["expected_section"],
        index_MP,
        query_embeddings_MP[i],
        k=5
    )

    results_mp.append({
        "id": q["id"],
        "question": q["question"],
        "expected_section": q["expected_section"],
        **result
    })

In [ ]:
total = len(results_minilm)

hit1 = sum(r["hit@1"] for r in results_minilm)
hit3 = sum(r["hit@3"] for r in results_minilm)
hit5 = sum(r["hit@5"] for r in results_minilm)

print("MiniLM Retrieval Evaluation")
print("----------------------------")

print("Total Questions:", total)

print("Hit@1:", hit1 / total)
print("Hit@3:", hit3 / total)
print("Hit@5:", hit5 / total)

MiniLM Retrieval Evaluation
----------------------------
Total Questions: 30
Hit@1: 0.43333333333333335
Hit@3: 0.6333333333333333
Hit@5: 0.7


In [ ]:
print("Hit@1:", round(hit1 / total * 100, 2), "%")
print("Hit@3:", round(hit3 / total * 100, 2), "%")
print("Hit@5:", round(hit5 / total * 100, 2), "%")

Hit@1: 43.33 %
Hit@3: 63.33 %
Hit@5: 70.0 %


In [ ]:
for r in results_minilm[:5]:

    print("\n" + "=" * 60)

    print("ID:", r["id"])
    print("Question:", r["question"])
    print("Expected:", r["expected_section"])

    print("Retrieved:", r["retrieved_indices"])

    print("Distances:", r["distances"])

    print("Hit@1:", r["hit@1"])
    print("Hit@3:", r["hit@3"])
    print("Hit@5:", r["hit@5"])


ID: Q01
Question: What is Retrieval-Augmented Generation?
Expected: RAG
Retrieved: [0, 12, 11, 29, 13]
Distances: [0.8344131708145142, 0.9144470691680908, 0.9328814148902893, 0.9853886961936951, 1.0342328548431396]
Hit@1: True
Hit@3: True
Hit@5: True

ID: Q02
Question: How does RAG use an external knowledge base?
Expected: RAG
Retrieved: [0, 2, 26, 28, 3]
Distances: [0.5612514615058899, 0.7818684577941895, 0.7957755327224731, 0.8689990639686584, 0.9147319793701172]
Hit@1: True
Hit@3: True
Hit@5: True

ID: Q03
Question: What are the main stages of a typical RAG pipeline?
Expected: RAG
Retrieved: [1, 3, 11, 0, 2]
Distances: [0.9756208062171936, 1.0740584135055542, 1.1959331035614014, 1.2502195835113525, 1.2541382312774658]
Hit@1: False
Hit@3: True
Hit@5: True

ID: Q04
Question: Why can RAG reduce hallucinations?
Expected: RAG
Retrieved: [2, 19, 20, 3, 0]
Distances: [0.8922187089920044, 0.9960722327232361, 1.3356208801269531, 1.3443355560302734, 1.4207518100738525]
Hit@1: False
Hit@3: Fa

In [ ]:
results_mp = []

for i, q in enumerate(questions):

    result = evaluate_question(
        q["question"],
        q["expected_section"],
        index_MP,
        query_embeddings_MP[i],
        k=5
    )

    results_mp.append({
        "id": q["id"],
        "question": q["question"],
        "expected_section": q["expected_section"],
        **result
    })

In [ ]:
total = len(results_mp)

hit1 = sum(r["hit@1"] for r in results_mp)
hit3 = sum(r["hit@3"] for r in results_mp)
hit5 = sum(r["hit@5"] for r in results_mp)

print("MPNet Retrieval Evaluation")
print("--------------------------")

print("Hit@1:", round(hit1 / total * 100, 2), "%")
print("Hit@3:", round(hit3 / total * 100, 2), "%")
print("Hit@5:", round(hit5 / total * 100, 2), "%")

MPNet Retrieval Evaluation
--------------------------
Hit@1: 43.33 %
Hit@3: 70.0 %
Hit@5: 76.67 %


# **Final COmparison**

In [ ]:
print("\nEmbedding Model Comparison")
print("==========================")

print("MiniLM")
print("Hit@1:", round(
    sum(r["hit@1"] for r in results_minilm) / len(results_minilm) * 100, 2
), "%")

print("Hit@3:", round(
    sum(r["hit@3"] for r in results_minilm) / len(results_minilm) * 100, 2
), "%")

print("Hit@5:", round(
    sum(r["hit@5"] for r in results_minilm) / len(results_minilm) * 100, 2
), "%")


print("\nMPNet")
print("Hit@1:", round(
    sum(r["hit@1"] for r in results_mp) / len(results_mp) * 100, 2
), "%")

print("Hit@3:", round(
    sum(r["hit@3"] for r in results_mp) / len(results_mp) * 100, 2
), "%")

print("Hit@5:", round(
    sum(r["hit@5"] for r in results_mp) / len(results_mp) * 100, 2
), "%")


Embedding Model Comparison
MiniLM
Hit@1: 43.33 %
Hit@3: 63.33 %
Hit@5: 70.0 %

MPNet
Hit@1: 43.33 %
Hit@3: 70.0 %
Hit@5: 76.67 %


# **Save the comparison**

In [ ]:
import pandas as pd

minilm_df = pd.DataFrame(results_minilm)
mpnet_df = pd.DataFrame(results_mp)

minilm_df.to_csv("minilm_retrieval_results.csv", index=False)
mpnet_df.to_csv("mpnet_retrieval_results.csv", index=False)

print("Results saved successfully.")

Results saved successfully.


In [ ]:
summary = pd.DataFrame({
    "Model": ["MiniLM", "MPNet"],
    "Hit@1": [
        sum(r["hit@1"] for r in results_minilm) / len(results_minilm) * 100,
        sum(r["hit@1"] for r in results_mp) / len(results_mp) * 100
    ],
    "Hit@3": [
        sum(r["hit@3"] for r in results_minilm) / len(results_minilm) * 100,
        sum(r["hit@3"] for r in results_mp) / len(results_mp) * 100
    ],
    "Hit@5": [
        sum(r["hit@5"] for r in results_minilm) / len(results_minilm) * 100,
        sum(r["hit@5"] for r in results_mp) / len(results_mp) * 100
    ]
})

summary.to_csv("retrieval_summary.csv", index=False)

In [ ]:
# from google.colab import files

# files.download("minilm_retrieval_results.csv")
# files.download("mpnet_retrieval_results.csv")
# files.download("retrieval_summary.csv")

In [ ]:
ip = ''
while ip != 'no':
  query = input('Enter your query: ')

  # Get embeddings for the query
  query_LM_embedding = model_LM.encode(query, convert_to_numpy=True)
  query_MP_embedding = model_MP.encode(query, convert_to_numpy=True)

  k = 3 # Number of top results to retrieve

  # Retrieve using MiniLM index
  distances_LM, indices_LM = retrieve(index_LM, query_LM_embedding, k)
  print(f"\nMiniLM Top {k} retrieved chunks:")
  for i, rank_idx in enumerate(indices_LM):
    print(f'Rank {i + 1}:')
    print(chunks[rank_idx])

  # Retrieve using MPNet index
  distances_MP, indices_MP = retrieve(index_MP, query_MP_embedding, k)
  print(f"\nMPNet Top {k} retrieved chunks:")
  for i, rank_idx in enumerate(indices_MP):
    print(f'Rank {i + 1}:')
    print(chunks[rank_idx])

  ip = input('Do you want to continue? (yes/no): ')

Enter your query: What is Self RAG

MiniLM Top 3 retrieved chunks:
Rank 1:
s self-reflection into the RAG process.

A Self-RAG system can decide when retrieval is necessary, generate a response using retrieved evidence, evaluate its response, and revise the response when required.

A simplified Self-RAG process is:

User Query
→ Retrieval
→ Response Generation
→ Self-Critique
→ Evaluation
→ Additional Retrieval or Revision
→ Final Response

Self-RAG attempts to make the generation process more adaptive. Instead of always retrieving the same number of documents, the sys
Rank 2:
n algorithm.

15. Limitations of Existing RAG Approaches

Traditional RAG can fail when irrelevant information is retrieved.

CRAG attempts to correct poor retrieval but primarily focuses on the retrieval stage.

Self-RAG introduces self-reflection but relies on the model's ability to evaluate its own responses.

GraphRAG improves relationship-based retrieval but does not automatically guarantee that the final re